# Week 3 — Unsupervised Learning and Customer Segmentation
**Student Name:** Sindhu Patil | **Internship:** Data Science

## 1. Introduction
Unsupervised learning discovers hidden structures, groupings, and behavioral patterns within unlabeled datasets without prior target label supervision.

## 2. Objective
Segment the IBM Telco customer base into distinct behavioral cohorts using K-Means clustering. Determine the optimal number of clusters ($K$), profile segment characteristics, analyze observed churn rates, and formulate targeted retention strategies.

## 3. What is Unsupervised Learning?
Unlike supervised learning (which predicts known targets), unsupervised algorithms group unlabeled data based on feature similarity and distance metrics (e.g., Euclidean distance).

## 4. Why Customer Segmentation?
Treating all customers identically leads to inefficient marketing. Segmentation allows telecommunication providers to tailor contract offers, service bundles, and support strategies to high-risk or high-value customer segments.

## 5. Dataset Preparation
Loading the cleaned dataset (`data/processed/cleaned_telco_churn.csv`).

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.clustering import prepare_clustering_data, calculate_elbow_scores, calculate_silhouette_scores, train_kmeans, profile_clusters, apply_pca
from src.visualization import plot_elbow_curve, plot_silhouette_scores, plot_cluster_sizes, plot_cluster_pca_2d, plot_avg_tenure_by_cluster, plot_avg_monthly_charges_by_cluster, plot_churn_rate_by_cluster, plot_contract_distribution_by_cluster

df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')
print('Cleaned Dataset Shape:', df.shape)

## 6. Feature Selection (Excluding Churn & customerID)
Selecting behavioral and subscription features. `Churn` target and `customerID` identifier are strictly excluded to avoid data leakage.

In [ ]:
X_scaled, df_enc, scaler, feature_names = prepare_clustering_data(df)
print('Clustering Feature Matrix Shape:', X_scaled.shape)
print('Encoded Feature Names:', feature_names)

## 7. Data Preprocessing & Categorical Encoding
One-Hot Encoding multi-class categorical features (`Contract`, `InternetService`, `PaymentMethod`, etc.).

## 8. Feature Scaling (`StandardScaler`)
Standardizing features to zero mean and unit variance so high-magnitude variables (`TotalCharges`) do not artificially dominate Euclidean distance calculations.

## 9. Choosing the Number of Clusters ($K=2$ to $K=10$)
Testing cluster counts from $K=2$ through $K=10$.

In [ ]:
k_range = range(2, 11)
k_list, inertias = calculate_elbow_scores(X_scaled, k_range=k_range)
_, sil_scores = calculate_silhouette_scores(X_scaled, k_range=k_range)

k_results = pd.DataFrame({'K': k_list, 'Inertia': inertias, 'Silhouette_Score': sil_scores})
print(k_results.to_string(index=False))

## 10. Elbow Method Visualization

In [ ]:
plot_elbow_curve(k_list, inertias, '../outputs/figures/week3/elbow_method.png')
plt.figure(figsize=(7, 4))
plt.plot(k_list, inertias, 'bo-')
plt.title('Elbow Method (Inertia vs K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

## 11. Silhouette Score Visualization

In [ ]:
plot_silhouette_scores(k_list, sil_scores, '../outputs/figures/week3/silhouette_scores.png')
plt.figure(figsize=(7, 4))
plt.plot(k_list, sil_scores, 'rs--')
plt.title('Silhouette Score vs K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.show()

## 12. Final K-Means Model ($K=3$ Selection)
$K=3$ was selected because the Elbow reduction levels off, silhouette scores remain robust, and 3 distinct business cohorts emerge.

In [ ]:
kmeans_model, cluster_labels, model_path = train_kmeans(X_scaled, n_clusters=3, model_filename='../outputs/models/kmeans_model.joblib')
df['Cluster'] = cluster_labels

## 13. Cluster Assignment & Size Summary

In [ ]:
summary_df, df_clustered, cluster_map = profile_clusters(df, cluster_labels)
summary_df[['Cluster', 'Segment_Name', 'Customer_Count', 'Percentage', 'Avg_Tenure', 'Avg_Monthly_Charges', 'Churn_Rate']]

## 14. Cluster Size & Feature Visualizations

In [ ]:
plot_cluster_sizes(summary_df, '../outputs/figures/week3/cluster_sizes.png')
plot_avg_tenure_by_cluster(summary_df, '../outputs/figures/week3/avg_tenure_by_cluster.png')
plot_avg_monthly_charges_by_cluster(summary_df, '../outputs/figures/week3/avg_monthly_charges_by_cluster.png')

## 15. PCA 2D Projection Visualization

In [ ]:
X_pca, pca_obj, exp_var = apply_pca(X_scaled, n_components=2)
centroids_pca = pca_obj.transform(kmeans_model.cluster_centers_)
names = [cluster_map[i] for i in range(3)]
plot_cluster_pca_2d(X_pca, cluster_labels, centroids_pca, names, '../outputs/figures/week3/cluster_pca_2d.png')

## 16. Cluster Profiling (Numerical & Categorical Breakdown)

In [ ]:
print('--- Contract Type Distribution per Cluster (%) ---')
print((pd.crosstab(df['Cluster'], df['Contract'], normalize='index') * 100).round(2))
print('
--- Internet Service Distribution per Cluster (%) ---')
print((pd.crosstab(df['Cluster'], df['InternetService'], normalize='index') * 100).round(2))

## 17. Observed Churn Rate Analysis by Cluster

In [ ]:
plot_churn_rate_by_cluster(summary_df, '../outputs/figures/week3/churn_rate_by_cluster.png')
plot_contract_distribution_by_cluster(df_clustered, cluster_map, '../outputs/figures/week3/contract_distribution_by_cluster.png')

## 18. Cluster Interpretation & Naming
- **Cluster 0 (High-Value Loyal Customers):** 2,337 customers (33.18%), Avg Tenure = 56.41 mos, Avg Monthly Bill = $89.68, Churn Rate = **15.36%**.
- **Cluster 1 (New High-Charge At-Risk Customers):** 3,180 customers (45.15%), Avg Tenure = 14.13 mos, Avg Monthly Bill = $67.99, Churn Rate = **43.19%**.
- **Cluster 2 (Long-Term Budget Customers):** 1,526 customers (21.67%), Avg Tenure = 32.22 mos, Avg Monthly Bill = $21.08, Churn Rate = **7.40%**.

## 19. Business Implications & Action Plans
- **High-Value Loyal Customers (Cluster 0):** Implement VIP loyalty perks, priority support, and long-term contract renewal incentives.
- **New High-Charge At-Risk Customers (Cluster 1):** Provide 0–12 month onboarding support, bundle free tech support, and offer contract conversion discounts.
- **Long-Term Budget Customers (Cluster 2):** Target with low-cost digital value add-ons without increasing baseline pricing.

## 20. Limitations
- K-Means assumes spherical cluster geometry.
- 2D PCA projection compresses 27-dimensional feature variance to 57.89%.

## 21. Conclusion
K-Means clustering ($K=3$) partitioned the customer base into 3 distinct cohorts, identifying Cluster 1 as the primary at-risk segment (43.19% churn rate).